<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Machine-Learning/06-evaluation-experimental-design.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Machine Learning guideline](Machine Learning.html)


## **Evaluation, Experimental Design, and Model Selection**

Training produces a fitted model; evaluation determines what can responsibly be claimed about it. The central quantity is not a score already observed on a convenient dataset, but the model's expected behavior on future units drawn from a stated **target population**. If the future distribution is $P_{\mathrm{target}}(X,Y)$ and a fitted predictor is $f$, a generic target risk is

$$
R_{\mathrm{target}}(f)
=\mathbb{E}_{(X,Y)\sim P_{\mathrm{target}}}
\left[L\bigl(Y,f(X)\bigr)\right].
$$

The expectation is unknown. A test set, cross-validation procedure, or bootstrap estimates it from finite observations. Consequently, an evaluation result depends on more than a metric name: it depends on the population, sampling unit, split rule, preprocessing pipeline, model-selection procedure, decision threshold, and uncertainty calculation.

Three questions should therefore be separated:

- **What is being predicted?** Define the unit, outcome, prediction horizon, and operating environment.
- **What is being selected?** Features, preprocessing, algorithms, hyperparameters, thresholds, and stopping rules all belong to model selection.
- **What is being estimated?** A fixed model's performance, the performance of an entire training procedure, subgroup behavior, or deployment utility are different estimands.

Chapter 05 distinguished a training objective from an evaluation metric. This chapter adds a second distinction: **validation data guide decisions, whereas test data estimate the result after those decisions are frozen**. Repeatedly inspecting a test score turns the test set into another validation set, even when no gradient is computed on it.

![Evaluation proceeds from a target population through a structure-aware split and a validation-based selection loop; the test set is used only after the decision rule is frozen.](assets/evaluation-lifecycle.svg){fig-align="center" width="100%" fig-alt="Controlled evaluation lifecycle with target population, structured split, selection loop, frozen decision rule, one final test, and deployment monitoring"}

The lifecycle also explains why deployment monitoring does not replace evaluation. A historical test set answers how the frozen procedure behaved under a historical sampling process. Monitoring asks whether performance, calibration, latency, and subgroup behavior remain acceptable after the data-generating process changes.


### **Evaluation Protocols and Data Splits**

An **evaluation protocol** is the complete rule that converts available data into fitted candidates and an out-of-sample estimate. It includes the split unit, resampling scheme, preprocessing boundaries, search space, metric, random seeds, and the point at which the test set becomes locked. A protocol should imitate the information and constraints available at deployment.

#### **Training, Validation, and Test Sets**

The three subsets have different jobs:

| Subset | Permitted use | Typical decisions |
|---|---|---|
| Training | Estimate model parameters and fit data-dependent transforms | coefficients, tree splits, vocabulary, imputation values, scaling statistics |
| Validation | Compare candidate procedures | model family, hyperparameters, features, threshold, early stopping |
| Test | Estimate the final frozen procedure once | point estimate, uncertainty interval, final error analysis for reporting |

Let $A_\lambda$ denote a training algorithm with hyperparameters $\lambda$. Validation selects

$$
\widehat\lambda
=\arg\min_{\lambda\in\Lambda}
\widehat R_{\mathrm{val}}
\left(A_\lambda(D_{\mathrm{train}})\right),
$$

and the test set then evaluates $A_{\widehat\lambda}$. Because $\widehat\lambda$ depends on validation outcomes, the validation score of the winner is optimistically biased. The test set remains useful only when it has not influenced $\Lambda$, preprocessing, threshold choice, or the decision to keep experimenting.

The split must occur at the unit that will be independent in deployment. Images from the same patient, messages from the same user, frames from the same video, and measurements from the same machine are not independent rows. Splitting their rows independently allows near-duplicates or entity fingerprints to cross the boundary.

<details>
<summary><strong>Python example: a validation set selects the pipeline; the test set evaluates it once</strong></summary>

```python
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)

# First isolate the final test set. The second split creates validation data.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=7
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=7
)

candidates = []
for C in [0.01, 0.1, 1.0, 10.0]:
    # Scaling is inside the fitted pipeline, so it learns from training data only.
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(C=C, max_iter=2000, random_state=7),
    )
    model.fit(X_train, y_train)
    val_prob = model.predict_proba(X_val)[:, 1]
    candidates.append((log_loss(y_val, val_prob), C, model))

val_loss, best_C, selected_model = min(candidates, key=lambda row: row[0])
test_prob = selected_model.predict_proba(X_test)[:, 1]
test_pred = (test_prob >= 0.5).astype(int)

print("split sizes:", len(y_train), len(y_val), len(y_test))
print("selected C:", best_C, "validation log loss:", round(val_loss, 4))
print("locked-test log loss:", round(log_loss(y_test, test_prob), 4))
print("locked-test accuracy:", round(accuracy_score(y_test, test_pred), 4))
```

</details>

#### **Holdout Evaluation**

A **holdout** protocol makes one train/test split, or one train/validation/test split. It is simple, computationally inexpensive, and easy to reproduce. It works well when data are abundant and the split mirrors deployment, but its answer can depend strongly on which observations happen to enter the test set.

For independent binary outcomes and a fixed classifier, test accuracy $\widehat p$ on $m$ cases has the approximate standard error

$$
\operatorname{SE}(\widehat p)
\approx\sqrt{\frac{\widehat p(1-\widehat p)}{m}}.
$$

This expression captures only test-sample uncertainty under an idealized Bernoulli model. It does not include variation from retraining on a different training sample, model selection, correlated units, temporal drift, or label noise. A large test set reduces sampling noise; it does not repair a mismatched population or leakage.

Repeated holdout can reveal split sensitivity by rerunning the **entire fitting procedure** across several random splits. However, reporting the best split is invalid, and the resulting scores are correlated because their training and test sets overlap.

<details>
<summary><strong>Python example: repeated holdout exposes split-to-split variation</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=800,
    n_features=12,
    n_informative=5,
    class_sep=0.8,
    flip_y=0.08,
    random_state=11,
)
splitter = StratifiedShuffleSplit(n_splits=30, test_size=0.20, random_state=11)
scores = []

for train_idx, test_idx in splitter.split(X, y):
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, random_state=11),
    )
    model.fit(X[train_idx], y[train_idx])
    scores.append(accuracy_score(y[test_idx], model.predict(X[test_idx])))

scores = np.asarray(scores)
binomial_se = np.sqrt(scores.mean() * (1 - scores.mean()) / (0.20 * len(y)))
print("mean accuracy:", round(scores.mean(), 3))
print("standard deviation across splits:", round(scores.std(ddof=1), 3))
print("observed range:", tuple(np.round([scores.min(), scores.max()], 3)))
print("single-test binomial SE approximation:", round(binomial_se, 3))
```

</details>

#### **K-Fold and Stratified Cross-Validation**

In $K$-fold cross-validation, the sample is partitioned into disjoint folds $I_1,\ldots,I_K$. For each fold, the model is fitted on all other folds and scored on $I_k$:

$$
\widehat R_{\mathrm{CV}}
=\frac{1}{K}\sum_{k=1}^{K}
\widehat R_{I_k}\left(A(D\setminus I_k)\right).
$$

Every observation is evaluated out of sample once, while each fitted model uses $(K-1)/K$ of the data. Larger $K$ reduces the training-size mismatch but increases computation and often increases correlation among fitted models. Fold scores are not independent replicates, so their standard deviation is useful as a stability description but is not automatically a valid confidence interval for generalization error.

**Stratified** cross-validation approximately preserves class proportions in each fold. It prevents a rare class from disappearing from a fold and stabilizes metrics such as recall. It does not protect groups, chronology, duplicates, or latent entities. Repeated stratified cross-validation averages over several partitions when split variability matters.

![K-fold cross-validation rotates the held-out fold while training on the remainder.](assets/cv-kfold.png){fig-align="center" width="72%" fig-alt="Visualization of K-fold cross-validation train and test indices"}

![Stratified K-fold cross-validation maintains the class mixture across folds.](assets/cv-stratified.png){fig-align="center" width="72%" fig-alt="Visualization of stratified K-fold cross-validation with class labels"}

*Images: [scikit-learn, Visualizing cross-validation behavior](https://scikit-learn.org/stable/auto_examples/model_selection/plot_cv_indices.html).*

<details>
<summary><strong>Python example: stratification stabilizes minority prevalence across folds</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=500,
    n_features=10,
    weights=[0.92, 0.08],
    n_informative=4,
    random_state=19,
)
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=19),
)

splitters = {
    "KFold": KFold(n_splits=5, shuffle=True, random_state=19),
    "StratifiedKFold": StratifiedKFold(n_splits=5, shuffle=True, random_state=19),
}

for name, cv in splitters.items():
    minority_rates = [y[test].mean() for _, test in cv.split(X, y)]
    scores = cross_val_score(model, X, y, cv=cv, scoring="balanced_accuracy")
    print(name)
    print("  minority rate per fold:", np.round(minority_rates, 3))
    print("  balanced accuracy:     ", np.round(scores, 3))
```

</details>

#### **Grouped and Time-Series Evaluation**

When rows share an entity, **grouped evaluation** assigns every row from one group to the same fold. If $g_i$ is a patient, user, household, document, or device identifier, the constraint is

$$
g_i=g_j\quad\Longrightarrow\quad
i\text{ and }j\text{ cannot appear on opposite sides of a split}.
$$

This usually yields a harder but more relevant question: can the model generalize to a new group rather than recognize a group seen during training? If deployment predicts future observations for already-known entities, a temporal within-entity design may instead be appropriate. The split follows the deployment question, not a universal recipe.

![Group-based cross-validation keeps all observations from an entity together.](assets/cv-grouped.png){fig-align="center" width="72%" fig-alt="Visualization of grouped cross-validation where group rows remain in one fold"}

For time-ordered data, random splitting leaks future conditions into the past. A forward-chaining split trains on times before the validation interval:

$$
\max(t_i:i\in D_{\mathrm{train}})
<\min(t_j:j\in D_{\mathrm{validation}}).
$$

An **expanding window** retains all earlier observations; a **rolling window** retains only a recent history. A gap or embargo between train and validation can prevent leakage when features use delayed labels, overlapping windows, or slowly resolved outcomes. Evaluation should also match the forecast horizon: predicting one hour ahead and one month ahead are different tasks.

![Time-series cross-validation expands the training prefix and evaluates only later observations.](assets/cv-time-series.png){fig-align="center" width="72%" fig-alt="Visualization of forward-only time-series cross-validation"}

*Images: [scikit-learn, Visualizing cross-validation behavior](https://scikit-learn.org/stable/auto_examples/model_selection/plot_cv_indices.html).*

<details>
<summary><strong>Python example: row-wise cross-validation leaks group identity</strong></summary>

```python
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

rng = np.random.default_rng(23)
n_groups, rows_per_group = 120, 4
groups = np.repeat(np.arange(n_groups), rows_per_group)

# Each group has a stable fingerprint and a random group-level label.
centres = rng.normal(size=(n_groups, 8))
X = centres[groups] + rng.normal(scale=0.02, size=(len(groups), 8))
group_labels = rng.integers(0, 2, size=n_groups)
y = group_labels[groups]

def evaluate(splitter, uses_groups):
    scores = []
    split_args = (X, y, groups) if uses_groups else (X, y)
    for train, test in splitter.split(*split_args):
        model = KNeighborsClassifier(n_neighbors=1).fit(X[train], y[train])
        scores.append(accuracy_score(y[test], model.predict(X[test])))
    return np.mean(scores)

row_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=23)
group_cv = GroupKFold(n_splits=5)

print("row-wise CV accuracy: ", round(evaluate(row_cv, False), 3))
print("group-wise CV accuracy:", round(evaluate(group_cv, True), 3))
```

</details>

<details>
<summary><strong>Python example: inspect forward-only training and validation windows</strong></summary>

```python
import numpy as np
from sklearn.model_selection import TimeSeriesSplit

time = np.arange(24)
splitter = TimeSeriesSplit(n_splits=4, test_size=4, gap=1)

for fold, (train, test) in enumerate(splitter.split(time), start=1):
    print(
        f"fold {fold}: train {train[0]:02d}-{train[-1]:02d}, "
        f"gap {train[-1] + 1:02d}, test {test[0]:02d}-{test[-1]:02d}"
    )
```

</details>

#### **Nested Cross-Validation**

Ordinary cross-validation estimates a fixed procedure only when all choices are already specified. If the same folds are used to search many configurations and to report the winner's score, selection favors configurations that benefited from random validation noise.

**Nested cross-validation** separates these roles:

1. The **inner loop** searches hyperparameters using only the outer training partition.
2. The selected inner procedure is refitted on that entire outer training partition.
3. The **outer fold** evaluates the selected procedure on data untouched by the inner search.
4. Outer scores are aggregated to estimate the performance of the complete selection procedure.

If $I_k$ is outer test fold $k$, the configuration is itself fold-specific,

$$
\widehat\lambda_k
=\arg\min_{\lambda\in\Lambda}
\widehat R_{\mathrm{inner},k}(\lambda),
$$

and $A_{\widehat\lambda_k}(D\setminus I_k)$ is evaluated on $I_k$. Nested CV estimates a **procedure that selects a model**, not the performance of one universally fixed hyperparameter value. After evaluation, a final model may be selected and fitted on all development data for deployment.

![Nested cross-validation places hyperparameter selection inside every outer training fold.](assets/nested-cross-validation.png){fig-align="center" width="72%" fig-alt="Comparison of nested and non-nested cross-validation scores"}

*Image: [scikit-learn, Nested versus non-nested cross-validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_nested_cross_validation_iris.html). The plotted difference illustrates selection optimism; its magnitude is dataset- and search-dependent.*

<details>
<summary><strong>Python example: estimate an SVM selection procedure with nested cross-validation</strong></summary>

```python
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score
from sklearn.svm import SVC

X, y = load_iris(return_X_y=True)
grid = {
    "C": [0.1, 1.0, 10.0, 100.0],
    "gamma": [0.001, 0.01, 0.1, 1.0],
}
differences = []

for seed in range(10):
    inner = KFold(n_splits=4, shuffle=True, random_state=seed)
    outer = KFold(n_splits=4, shuffle=True, random_state=100 + seed)

    # The same folds select and report the winner in the non-nested estimate.
    non_nested = GridSearchCV(SVC(), grid, cv=outer, scoring="accuracy", n_jobs=1)
    non_nested.fit(X, y)

    # Every outer training fold runs an independent inner search.
    nested = GridSearchCV(SVC(), grid, cv=inner, scoring="accuracy", n_jobs=1)
    nested_score = cross_val_score(
        nested, X, y, cv=outer, scoring="accuracy", n_jobs=1
    ).mean()
    differences.append(non_nested.best_score_ - nested_score)

print("selection optimism in 10 repetitions:", np.round(differences, 4))
print("mean non-nested minus nested score:   ", round(np.mean(differences), 4))
print("individual repetitions may be zero or negative; the bias is an expectation")
```

</details>

**Protocol comparison.** Holdout is economical but sensitive to one split. Repeated holdout describes split sensitivity. K-fold cross-validation uses limited data efficiently. Stratification stabilizes class composition; grouping and temporal splits encode dependence and chronology. Nested cross-validation is warranted when the performance of a nontrivial selection process must be estimated without a separate large test set. None compensates for a sample that fails to represent the intended deployment population.


### **Task-Specific Evaluation Metrics**

A metric is a compressed answer to a specific question. Before choosing one, identify the prediction object and the action it supports:

- a **label metric** evaluates classifications after a particular threshold or decision rule;
- a **ranking metric** evaluates whether relevant cases tend to receive higher scores across thresholds;
- a **probability metric** evaluates the entire predictive distribution and rewards calibrated uncertainty;
- a **decision metric** combines predictions with costs, constraints, or utility.

The same model can be excellent at one level and poor at another. A monotonic transformation preserves rankings and therefore ROC AUC, yet it can destroy probability calibration. A well-calibrated model can still make poor actions if its threshold ignores asymmetric costs. Metric definitions must therefore include the positive class, threshold, averaging rule, sampling unit, and denominator.

![Classification metrics correspond to different output levels and should be selected from the intended use.](assets/classification-metric-map.svg){fig-align="center" width="100%" fig-alt="Map from classifier scores to label, ranking, probability, and decision metrics"}

#### **Classification Metrics**

For binary classification, let the positive class be the event of interest. The confusion matrix counts true positives ($TP$), false positives ($FP$), true negatives ($TN$), and false negatives ($FN$). Common threshold-dependent measures are

$$
\begin{aligned}
\mathrm{accuracy} &= \frac{TP+TN}{TP+TN+FP+FN},\\
\mathrm{precision} &= \frac{TP}{TP+FP},\\
\mathrm{recall\;(sensitivity)} &= \frac{TP}{TP+FN},\\
\mathrm{specificity} &= \frac{TN}{TN+FP},\\
F_\beta &= (1+\beta^2)
\frac{\mathrm{precision}\,\mathrm{recall}}
{\beta^2\mathrm{precision}+\mathrm{recall}}.
\end{aligned}
$$

Accuracy weights every case equally, so a dominant class can conceal failure on a rare class. **Balanced accuracy** averages sensitivity across classes. $F_1$ balances precision and recall but omits true negatives; $F_\beta$ gives recall more weight when $\beta>1$. The Matthews correlation coefficient (MCC) uses all four cells and remains informative under imbalance, but no scalar says which error type is operationally acceptable. Undefined ratios, such as precision when no positive prediction is made, must be reported explicitly rather than silently converted into evidence of good performance.

<details>
<summary><strong>Python example: high accuracy can coexist with zero minority-class recall</strong></summary>

```python
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
)

y_true = np.array([0] * 95 + [1] * 5)
predictions = {
    "always negative": np.zeros(100, dtype=int),
    "minority-aware": np.array([0] * 90 + [1] * 5 + [0, 1, 1, 1, 1]),
}

for name, y_pred in predictions.items():
    print("\n" + name)
    print("confusion matrix:\n", confusion_matrix(y_true, y_pred))
    print("accuracy:         ", round(accuracy_score(y_true, y_pred), 3))
    print("balanced accuracy:", round(balanced_accuracy_score(y_true, y_pred), 3))
    print("precision:        ", round(precision_score(y_true, y_pred, zero_division=0), 3))
    print("recall:           ", round(recall_score(y_true, y_pred), 3))
    print("F1 / MCC:         ", round(f1_score(y_true, y_pred), 3),
          round(matthews_corrcoef(y_true, y_pred), 3))
```

</details>

When a classifier emits a continuous score, varying the threshold traces trade-offs:

- The **ROC curve** plots true-positive rate against false-positive rate. ROC AUC equals the probability that a randomly selected positive receives a higher score than a randomly selected negative, with half credit for ties.
- The **precision-recall (PR) curve** plots precision against recall. It makes positive-class performance and prevalence visible, which is especially useful when positives are rare. A no-skill random ranking has expected precision equal to positive prevalence.
- **Average precision (AP)** summarizes precision at the recall increments induced by ranked examples. It is not generally identical to trapezoidal area under an interpolated PR curve.

![A precision-recall curve reveals the trade-off between retrieving more positive cases and maintaining precision.](assets/precision-recall-curve.png){fig-align="center" width="72%" fig-alt="Precision-recall curve with average precision annotation"}

*Image: [scikit-learn, Precision-Recall](https://scikit-learn.org/stable/auto_examples/model_selection/plot_precision_recall.html).*

Ranking metrics ignore whether a score of $0.8$ really means an 80% event rate. Proper probability scores retain that information. For binary labels $y_i\in\{0,1\}$ and probabilities $p_i$,

$$
\mathrm{log\ loss}
=-\frac{1}{n}\sum_{i=1}^{n}
\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right],
$$

$$
\mathrm{Brier\ score}
=\frac{1}{n}\sum_{i=1}^{n}(p_i-y_i)^2.
$$

Both are **proper scoring rules**: in expectation, honest probabilities minimize the score. Log loss penalizes confident mistakes especially strongly. Brier score has a bounded quadratic scale for binary outcomes and combines calibration and resolution, so it is not a pure calibration metric.

<details>
<summary><strong>Python example: identical ranking can hide very different probability quality</strong></summary>

```python
import numpy as np
from scipy.special import expit, logit
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score

rng = np.random.default_rng(37)
base_probability = rng.uniform(0.05, 0.95, size=2000)
y = rng.binomial(1, base_probability)

# A monotonic logit transformation preserves every pairwise ranking.
probabilities = {
    "well scaled": base_probability,
    "overconfident": expit(3.0 * logit(base_probability)),
}

for name, p in probabilities.items():
    print(
        f"{name:13s}",
        "ROC AUC =", round(roc_auc_score(y, p), 3),
        "AP =", round(average_precision_score(y, p), 3),
        "log loss =", round(log_loss(y, p), 3),
        "Brier =", round(brier_score_loss(y, p), 3),
    )
```

</details>

For multiclass problems, first compute a one-versus-rest statistic for each class and then state the averaging rule:

- **macro** averaging gives every class equal weight and exposes weak rare-class behavior;
- **weighted macro** weights each class by its support and can be dominated by common classes;
- **micro** averaging pools all decisions before computing the metric and emphasizes common observations;
- **per-class** values preserve the most diagnostic information.

Top-$k$ accuracy is appropriate only when downstream users genuinely inspect $k$ candidates. Multilabel tasks require further choices between example-wise and label-wise averaging; subset accuracy is extremely strict because every label for an example must match.

<details>
<summary><strong>Python example: micro, macro, and weighted F1 answer different questions</strong></summary>

```python
import numpy as np
from sklearn.metrics import classification_report, f1_score

# Class 0 is common; class 2 is rare and completely missed.
y_true = np.array([0] * 12 + [1] * 5 + [2] * 3)
y_pred = np.array([0] * 11 + [1] + [1] * 4 + [0] + [0, 1, 1])

for average in ["micro", "macro", "weighted"]:
    print(f"{average:8s} F1:", round(f1_score(y_true, y_pred, average=average), 3))

print("\nPer-class report:")
print(classification_report(y_true, y_pred, digits=3, zero_division=0))
```

</details>

#### **Regression Metrics**

Regression errors retain magnitude and units. For residual $e_i=y_i-\widehat y_i$,

$$
\mathrm{MAE}=\frac{1}{n}\sum_i|e_i|,
\qquad
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_i e_i^2}.
$$

MAE describes a typical absolute miss and grows linearly with large errors. RMSE has the target's units but, because it squares residuals before averaging, emphasizes occasional large misses. Which is preferable depends on the cost curve and noise assumptions, not on which produces a more flattering number. Median absolute error is still more resistant to a small number of extreme cases.

The coefficient of determination compares squared error with a mean-prediction baseline:

$$
R^2
=1-\frac{\sum_i(y_i-\widehat y_i)^2}
{\sum_i(y_i-\overline y)^2}.
$$

$R^2=0$ matches that baseline on the evaluated sample; $R^2<0$ is possible and means the model is worse. It is not a percentage of predictions that are correct, and comparisons across datasets with different target variance can be misleading.

Percentage errors such as MAPE are undefined at $y_i=0$ and explode near zero. RMSLE requires nonnegative values and emphasizes relative multiplicative error. For asymmetric decisions, quantile or **pinball loss** at level $\tau$,

$$
\rho_\tau(e)=
\begin{cases}
\tau e,&e\geq0,\\
(\tau-1)e,&e<0,
\end{cases}
$$

evaluates a predicted conditional quantile. Always report target units, a simple baseline, and residual structure alongside scalar summaries.

![Regression evaluation combines scalar errors with actual-versus-predicted and residual diagnostics.](assets/regression-diagnostics.svg){fig-align="center" width="100%" fig-alt="Actual versus predicted plot, heteroscedastic residual plot, and regression metric summaries"}

<details>
<summary><strong>Python example: one outlier affects MAE and RMSE differently</strong></summary>

```python
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_true = np.array([10, 12, 14, 16, 18, 20], dtype=float)
ordinary_prediction = np.array([11, 11, 15, 15, 19, 19], dtype=float)
one_large_miss = ordinary_prediction.copy()
one_large_miss[-1] = 35

for name, y_pred in [
    ("small errors", ordinary_prediction),
    ("one large miss", one_large_miss),
]:
    print(
        f"{name:14s}",
        "MAE =", round(mean_absolute_error(y_true, y_pred), 3),
        "RMSE =", round(mean_squared_error(y_true, y_pred) ** 0.5, 3),
        "R2 =", round(r2_score(y_true, y_pred), 3),
    )
```

</details>

Residuals diagnose failures hidden by averages. A residual-versus-prediction plot should be centered near zero without systematic curvature or a widening funnel. Curvature suggests missing nonlinear structure; variance that increases with scale suggests heteroscedasticity; temporal runs suggest drift or autocorrelation. Residuals should also be sliced by subgroup and target range because equal overall MAE does not imply equal conditional behavior.

<details>
<summary><strong>Python example: residual slices reveal scale-dependent error</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

rng = np.random.default_rng(41)
X = rng.uniform(0, 10, size=(800, 1))
noise = rng.normal(scale=0.25 + 0.35 * X[:, 0])
y = 2.0 + 1.8 * X[:, 0] + noise

model = LinearRegression().fit(X, y)
prediction = model.predict(X)
frame = pd.DataFrame({"prediction": prediction, "absolute_error": np.abs(y - prediction)})
frame["prediction_quartile"] = pd.qcut(frame["prediction"], 4, duplicates="drop")

print("overall MAE:", round(mean_absolute_error(y, prediction), 3))
print(frame.groupby("prediction_quartile", observed=True)["absolute_error"]
      .agg(["count", "mean"]).round(3))
```

</details>

#### **Clustering Metrics**

Clustering has no single ground truth objective. A partition may be useful for compression, market segmentation, anomaly discovery, or scientific interpretation, and each purpose implies different evidence.

Without labels, **internal metrics** compare compactness and separation. For sample $i$, let $a(i)$ be its mean distance to its own cluster and $b(i)$ the smallest mean distance to another cluster. Its silhouette value is

$$
s(i)=\frac{b(i)-a(i)}{\max\{a(i),b(i)\}}\in[-1,1].
$$

Values near 1 indicate separation, values near 0 indicate boundary points, and negative values suggest a closer alternative cluster. The Davies-Bouldin index compares within-cluster scatter with between-centroid separation; lower is better. Inertia decreases whenever more $k$-means clusters are added, so it cannot select $k$ by itself.

![Silhouette analysis shows both the average score and the distribution of sample-level values for each cluster.](assets/silhouette-analysis.png){fig-align="center" width="100%" fig-alt="Silhouette analysis for four K-means clusters beside the corresponding cluster scatter plot"}

*Image: [scikit-learn, Selecting the number of clusters with silhouette analysis](https://scikit-learn.org/stable/auto_examples/cluster/plot_kmeans_silhouette_analysis.html).*

When trusted external labels exist, adjusted Rand index (ARI), normalized mutual information, or V-measure compare partitions while ignoring arbitrary cluster identifiers. These labels may represent one semantic organization among several, so agreement is not always the objective. Stability under resampling, initialization, and feature perturbation is also important: an attractive but unstable partition is weak evidence. Internal distance metrics tend to favor their own geometric assumptions and may penalize meaningful nonconvex or unequal-density clusters.

<details>
<summary><strong>Python example: clustering metrics can prefer different numbers of clusters</strong></summary>

```python
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score, davies_bouldin_score, silhouette_score

X, true_labels = make_blobs(
    n_samples=600,
    centers=4,
    cluster_std=[0.55, 0.90, 0.65, 1.10],
    random_state=43,
)

for k in [2, 3, 4, 5, 6]:
    labels = KMeans(n_clusters=k, n_init=20, random_state=43).fit_predict(X)
    print(
        f"k={k}",
        "silhouette =", round(silhouette_score(X, labels), 3),
        "DB =", round(davies_bouldin_score(X, labels), 3),
        "ARI =", round(adjusted_rand_score(true_labels, labels), 3),
    )
```

</details>

#### **Dimensionality Reduction**

Dimensionality reduction can be evaluated against at least four different goals:

1. **Information retention:** explained variance and reconstruction error quantify what a projection preserves under a specified scale and loss.
2. **Geometry preservation:** trustworthiness asks whether neighbors in the embedding were also neighbors in the original space; continuity asks whether original neighbors remain nearby.
3. **Task utility:** held-out downstream performance, latency, memory, or sample efficiency test whether the representation serves its intended use.
4. **Stability and meaning:** sensitivity to seeds and samples, alignment across runs, and consistency with known factors determine whether an interpretation is reproducible.

![A reduced representation may be evaluated for retained information, geometry, downstream utility, or stability.](assets/representation-evaluation.svg){fig-align="center" width="100%" fig-alt="Encoder maps high-dimensional data to a representation that is evaluated along four distinct dimensions"}

For PCA, the cumulative explained-variance ratio after $d$ components is

$$
\frac{\sum_{j=1}^{d}\widehat\lambda_j}
{\sum_{j=1}^{p}\widehat\lambda_j},
$$

where $\widehat\lambda_j$ are ordered covariance eigenvalues. A high ratio does not guarantee class separation, local-neighborhood preservation, causal meaning, or fairness. For nonlinear visualization methods such as t-SNE and UMAP, visual cluster separation is exploratory evidence rather than a validated metric. Data-dependent scaling and dimensionality reduction must be fitted inside each training split when downstream generalization is evaluated.

<details>
<summary><strong>Python example: retained variance and downstream accuracy are different criteria</strong></summary>

```python
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import trustworthiness
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
X_scaled = StandardScaler().fit_transform(X)  # descriptive geometry only
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=47)

for dimensions in [2, 5, 10, 20]:
    pca = PCA(n_components=dimensions, random_state=47)
    embedded = pca.fit_transform(X_scaled)
    reconstructed = pca.inverse_transform(embedded)
    reconstruction_mse = np.mean((X_scaled - reconstructed) ** 2)

    # For predictive evaluation, scaling and PCA are refitted inside every fold.
    pipeline = make_pipeline(
        StandardScaler(),
        PCA(n_components=dimensions, random_state=47),
        LogisticRegression(max_iter=2000, random_state=47),
    )
    accuracy = cross_val_score(pipeline, X, y, cv=cv, scoring="accuracy").mean()
    print(
        f"d={dimensions:2d}",
        "variance =", round(pca.explained_variance_ratio_.sum(), 3),
        "reconstruction MSE =", round(reconstruction_mse, 3),
        "trustworthiness =", round(trustworthiness(X_scaled, embedded, n_neighbors=7), 3),
        "CV accuracy =", round(accuracy, 3),
    )
```

</details>

**Metric comparison.** Classification metrics differ by whether they consume labels, rankings, probabilities, or decisions. Regression metrics encode different error penalties and require residual diagnostics. Clustering metrics are conditional on a notion of geometric or semantic validity. Dimensionality reduction has multiple preservation targets. A defensible report uses a small complementary set chosen before inspecting results, preserves denominators and uncertainty, and explains what each number cannot establish.


### **Probability Calibration and Decision Thresholds**

A classifier commonly produces a score $s(x)$, which may be converted into a probability estimate $\widehat p(x)$ and then into an action. These are separate layers. Ranking asks whether positives receive larger scores; calibration asks whether stated probabilities match observed frequencies; thresholding asks which action has the best consequences.

#### **Reliability Diagrams and Calibration Error**

A binary predictor is perfectly calibrated when

$$
\Pr(Y=1\mid \widehat p=p)=p
$$

for every probability level with support. Among cases assigned probability near $0.7$, approximately 70% should be positive. Calibration is distribution-dependent: a model calibrated in one population may become miscalibrated after prevalence or conditional relationships change.

A **reliability diagram** partitions predictions into bins and plots the observed positive fraction against mean predicted probability. Points below the diagonal indicate overprediction of the positive event; points above indicate underprediction. Histograms of predicted probabilities should accompany the curve because apparently good calibration in a nearly empty bin is weak evidence.

![Reliability diagrams compare predicted probabilities with observed event frequencies, while histograms show where predictions occur.](assets/calibration-curves.png){fig-align="center" width="82%" fig-alt="Calibration curves and probability histograms for several classifiers"}

*Image: [scikit-learn, Probability calibration curves](https://scikit-learn.org/stable/auto_examples/calibration/plot_calibration_curve.html).*

A common summary is expected calibration error (ECE). With bins $B_1,\ldots,B_M$,

$$
\mathrm{ECE}
=\sum_{m=1}^{M}\frac{|B_m|}{n}
\left|\operatorname{acc}(B_m)-\operatorname{conf}(B_m)\right|,
$$

where $\operatorname{acc}(B_m)$ is the empirical event rate and $\operatorname{conf}(B_m)$ is the mean predicted probability. ECE is easy to interpret but depends on the number and placement of bins, discards within-bin variation, and has finite-sample bias. It should not replace proper scores such as log loss or Brier score.

Post-hoc calibration methods learn a mapping from raw scores to probabilities. **Platt scaling** fits a logistic map and is relatively low variance; **isotonic regression** fits a monotone stepwise map and is more flexible but needs more calibration data. The calibration map must be fitted on data that were not used to fit the base model, or inside a cross-validation procedure. Calibrating and evaluating on the same cases produces another form of optimism.

<details>
<summary><strong>Python example: fit isotonic calibration on a separate calibration split</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split

def expected_calibration_error(y, probability, bins=10):
    edges = np.linspace(0.0, 1.0, bins + 1)
    # Values equal to 1.0 remain in the final bin.
    membership = np.clip(np.digitize(probability, edges[1:-1]), 0, bins - 1)
    error = 0.0
    for index in range(bins):
        mask = membership == index
        if mask.any():
            error += mask.mean() * abs(y[mask].mean() - probability[mask].mean())
    return error

X, y = make_classification(
    n_samples=3000,
    n_features=20,
    n_informative=8,
    class_sep=0.8,
    random_state=53,
)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.50, stratify=y, random_state=53
)
X_cal, X_test, y_cal, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=53
)

base_model = RandomForestClassifier(
    n_estimators=200, min_samples_leaf=8, random_state=53, n_jobs=1
).fit(X_train, y_train)
calibration_score = base_model.predict_proba(X_cal)[:, 1]
test_score = base_model.predict_proba(X_test)[:, 1]

# Isotonic regression learns only from the dedicated calibration split.
calibrator = IsotonicRegression(out_of_bounds="clip").fit(calibration_score, y_cal)
calibrated_probability = np.clip(calibrator.predict(test_score), 1e-6, 1 - 1e-6)

for name, probability in [("raw", test_score), ("isotonic", calibrated_probability)]:
    print(
        f"{name:8s}",
        "ROC AUC =", round(roc_auc_score(y_test, probability), 3),
        "Brier =", round(brier_score_loss(y_test, probability), 3),
        "log loss =", round(log_loss(y_test, probability), 3),
        "ECE =", round(expected_calibration_error(y_test, probability), 3),
    )
```

</details>

Calibration can improve probability quality without improving ranking because a monotone map preserves order. Conversely, a highly discriminative model may be poorly calibrated. Report both when probabilities drive decisions.

#### **Threshold Selection under Unequal Costs**

The default threshold $0.5$ is justified only under particular assumptions about calibrated probabilities, equal error costs, and unconstrained decisions. Let $a\in\{0,1\}$ be an action and $C(a,y)$ its cost. The optimal action minimizes conditional expected cost:

$$
a^*(x)=\arg\min_a
\sum_y C(a,y)\Pr(Y=y\mid X=x).
$$

For binary decisions with zero cost for correct predictions, false-positive cost $c_{FP}$, and false-negative cost $c_{FN}$, predict positive when

$$
\widehat p(x)>
\frac{c_{FP}}{c_{FP}+c_{FN}}.
$$

This analytic threshold assumes calibrated probabilities and fixed costs. In practice, a threshold may instead be selected on validation data to minimize empirical cost, satisfy a capacity limit, achieve a minimum precision, or maximize recall subject to a false-positive budget. The chosen constraint must reflect deployment; using the test set to tune it contaminates the final estimate.

![Threshold tuning can substantially reduce an application-specific cost even when conventional classification summaries change in a less obvious way.](assets/threshold-cost-tuning.png){fig-align="center" width="100%" fig-alt="Cost-sensitive threshold tuning results and normalized business cost comparison"}

*Image: [scikit-learn, Post-tuning the decision threshold for cost-sensitive learning](https://scikit-learn.org/stable/auto_examples/model_selection/plot_cost_sensitive_learning.html).*

<details>
<summary><strong>Python example: select a cost-sensitive threshold on validation data</strong></summary>

```python
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(
    n_samples=4000,
    n_features=15,
    n_informative=7,
    weights=[0.88, 0.12],
    class_sep=0.9,
    random_state=59,
)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.50, stratify=y, random_state=59
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=59
)

model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=59),
).fit(X_train, y_train)
val_probability = model.predict_proba(X_val)[:, 1]
test_probability = model.predict_proba(X_test)[:, 1]

false_positive_cost, false_negative_cost = 1.0, 8.0

def cost_at_threshold(y_true, probability, threshold):
    prediction = probability >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, prediction).ravel()
    return (false_positive_cost * fp + false_negative_cost * fn) / len(y_true)

thresholds = np.linspace(0.01, 0.99, 99)
validation_costs = [cost_at_threshold(y_val, val_probability, t) for t in thresholds]
selected_threshold = thresholds[int(np.argmin(validation_costs))]

for threshold in [0.50, selected_threshold]:
    print(
        "threshold =", round(threshold, 2),
        "test cost per case =", round(cost_at_threshold(y_test, test_probability, threshold), 3),
    )
print("selected only from validation data:", round(selected_threshold, 2))
```

</details>

Thresholds are policy parameters rather than permanent properties of a classifier. They may need revision when prevalence, costs, capacity, or regulations change, while the underlying ranking model remains fixed. Such revision requires fresh validation evidence and monitoring for calibration drift.


### **Hyperparameter Search**

Model parameters are learned directly by minimizing a training objective; **hyperparameters** configure the learning procedure, representation, regularization, or architecture. Selecting them is an optimization problem over validation performance:

$$
\widehat\lambda
=\arg\max_{\lambda\in\Lambda}
\widehat U_{\mathrm{validation}}(\lambda).
$$

The search space $\Lambda$ is part of the experiment. Trying more configurations increases the chance of finding a genuinely good procedure, but also increases the chance of selecting a configuration that benefited from validation noise. All data-dependent preprocessing must remain inside the resampling pipeline, and final performance requires untouched test data or an outer cross-validation loop.

![Grid, random, Bayesian, and successive-halving searches allocate evaluation budget in different ways.](assets/search-strategies.svg){fig-align="center" width="100%" fig-alt="Comparison of four hyperparameter search strategies"}

#### **Grid and Random Search**

**Grid search** evaluates a Cartesian product of stated values. It is deterministic, easy to parallelize, and useful for a small number of discrete choices. Its cost grows exponentially with the number of hyperparameters: $m$ values for each of $d$ dimensions produce $m^d$ candidates. It also spends many trials varying unimportant dimensions while evaluating only a few distinct values of important continuous dimensions.

**Random search** samples configurations from distributions. With the same budget, it usually explores more distinct values along every dimension. The distributions encode prior scale knowledge: learning rates and regularization strengths often span orders of magnitude and should be sampled log-uniformly, while bounded fractions may be sampled uniformly or from beta distributions. Conditional parameters should be activated only for model families that use them.

The comparison is budget-sensitive. A fair study fixes the number of candidates, folds, metric, random seeds, preprocessing, and total resources. Search results should include the explored space, not just the winning configuration.

<details>
<summary><strong>Python example: compare grid and random search under explicit budgets</strong></summary>

```python
from scipy.stats import loguniform
from sklearn.datasets import make_classification
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X, y = make_classification(
    n_samples=900,
    n_features=16,
    n_informative=7,
    class_sep=0.9,
    random_state=61,
)
pipeline = make_pipeline(StandardScaler(), SVC())
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=61)

grid = GridSearchCV(
    pipeline,
    {
        "svc__C": [0.03, 0.3, 3.0, 30.0],
        "svc__gamma": [0.003, 0.03, 0.3],
    },
    cv=cv,
    scoring="roc_auc",
    n_jobs=1,
).fit(X, y)

random = RandomizedSearchCV(
    pipeline,
    {
        "svc__C": loguniform(1e-3, 1e2),
        "svc__gamma": loguniform(1e-4, 1.0),
    },
    n_iter=12,  # same number of candidates as the 4 x 3 grid
    cv=cv,
    scoring="roc_auc",
    random_state=61,
    n_jobs=1,
).fit(X, y)

for name, search in [("grid", grid), ("random", random)]:
    print(
        f"{name:6s}",
        "candidates =", len(search.cv_results_["params"]),
        "best CV AUC =", round(search.best_score_, 4),
        "best parameters =", search.best_params_,
    )
```

</details>

#### **Bayesian Optimization**

Bayesian optimization is designed for expensive black-box objectives. After observing trials $\mathcal D_t=\{(\lambda_i,u_i)\}_{i=1}^{t}$, it fits a **surrogate model** for validation utility and uses an **acquisition function** to choose the next configuration. Gaussian processes are common in low-dimensional continuous spaces; tree-structured Parzen estimators handle conditional mixed spaces more naturally.

Expected improvement for maximization is

$$
\operatorname{EI}(\lambda)
=\mathbb E\left[
\max\bigl(U(\lambda)-U_{\mathrm{best}}-\xi,0\bigr)
\mid\mathcal D_t
\right],
$$

where $\xi\geq0$ encourages exploration. A candidate is attractive when the surrogate predicts either a high mean or substantial uncertainty. The loop alternates between fitting the surrogate, maximizing the acquisition function, evaluating the selected configuration, and updating the observations.

Bayesian optimization is most useful when each trial is expensive and sequential feedback is affordable. It can struggle in very high dimensions, under noisy or nonstationary validation estimates, with many parallel workers, or when early trials poorly cover conditional choices. It optimizes the supplied validation signal, including its bias; it does not make a contaminated protocol valid.

<details>
<summary><strong>Python example: a one-dimensional Gaussian-process expected-improvement loop</strong></summary>

```python
import warnings
import numpy as np
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel

rng = np.random.default_rng(67)

def expensive_validation_score(x):
    # A stand-in for training and validating a model at hyperparameter x.
    signal = 0.82 - 0.025 * (x - 1.7) ** 2 + 0.035 * np.sin(2.8 * x)
    return float(signal + rng.normal(0, 0.004))

observed_x = np.array([-4.0, 0.0, 4.0])[:, None]
observed_y = np.array([expensive_validation_score(x) for x in observed_x[:, 0]])
candidate_grid = np.linspace(-5.0, 5.0, 401)[:, None]

kernel = ConstantKernel(1.0) * Matern(length_scale=1.5, nu=2.5) + WhiteKernel(1e-4)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for _ in range(7):
        surrogate = GaussianProcessRegressor(
            kernel=kernel,
            normalize_y=True,
            random_state=67,
            n_restarts_optimizer=1,
        ).fit(observed_x, observed_y)
        mean, std = surrogate.predict(candidate_grid, return_std=True)
        improvement = mean - observed_y.max() - 0.002
        z = np.divide(improvement, std, out=np.zeros_like(improvement), where=std > 0)
        expected_improvement = improvement * norm.cdf(z) + std * norm.pdf(z)
        next_x = candidate_grid[np.argmax(expected_improvement), 0]
        observed_x = np.vstack([observed_x, [[next_x]]])
        observed_y = np.append(observed_y, expensive_validation_score(next_x))

best = int(np.argmax(observed_y))
print("number of expensive evaluations:", len(observed_y))
print("best x:", round(float(observed_x[best, 0]), 3))
print("best observed validation score:", round(float(observed_y[best]), 4))
print("sequentially proposed x values:", np.round(observed_x[3:, 0], 3))
```

</details>

#### **Resource-Aware and Successive-Halving Methods**

Some configurations can be rejected after a small resource allocation. **Successive halving** starts with many candidates at low fidelity, retains a fraction of the strongest, increases their resources, and repeats. If the reduction factor is $\eta>1$, a stylized schedule uses approximately

$$
n_r\approx\frac{n_0}{\eta^r},
\qquad
b_r=b_0\eta^r,
$$

where $n_r$ is the number of surviving candidates and $b_r$ is resource per candidate at round $r$. Resources may be samples, epochs, boosting rounds, trees, or simulation time. Hyperband combines several halving brackets with different initial candidate counts and resource levels.

![Successive halving progressively allocates more resources to fewer surviving candidates.](assets/successive-halving.png){fig-align="center" width="76%" fig-alt="Number of candidates and resources across successive-halving iterations"}

*Image: [scikit-learn, Successive Halving Iterations](https://scikit-learn.org/stable/auto_examples/model_selection/plot_successive_halving_iterations.html).*

The key assumption is that low-resource performance is informative about full-resource performance. It fails when models learn at different rates, small subsets change class or group composition, or promising configurations start slowly. Every rung must still use valid validation data; an untouched outer test set cannot become the resource used for elimination.

<details>
<summary><strong>Python example: successive halving increases trees for surviving forests</strong></summary>

```python
from scipy.stats import randint
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingRandomSearchCV, StratifiedKFold

X, y = make_classification(
    n_samples=1000,
    n_features=18,
    n_informative=8,
    random_state=71,
)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=71)

search = HalvingRandomSearchCV(
    RandomForestClassifier(random_state=71, n_jobs=1),
    {
        "max_depth": randint(2, 16),
        "min_samples_leaf": randint(1, 12),
        "max_features": ["sqrt", "log2", None],
    },
    n_candidates=12,
    resource="n_estimators",
    min_resources=10,
    max_resources=80,
    factor=2,
    cv=cv,
    scoring="roc_auc",
    random_state=71,
    n_jobs=1,
).fit(X, y)

print("candidates by round:", search.n_candidates_)
print("trees by round:     ", search.n_resources_)
print("best CV AUC:        ", round(search.best_score_, 4))
print("best configuration: ", search.best_params_)
```

</details>

**Search comparison.** Grid search is transparent for small discrete spaces. Random search is a strong default for mixed continuous scales. Bayesian optimization learns where to evaluate next when trials are costly. Successive halving saves resources when early performance predicts later quality. In every case, search is repeated consultation with validation data; the breadth of that consultation belongs in the reported experimental design.


### **Uncertainty and Statistical Comparison**

A point estimate changes when the sampled test cases, training data, random seed, or selected hyperparameters change. These are different uncertainty sources. A valid interval or test must state which source is treated as random and preserve the dependence structure of the data.

![Cross-validated ROC curves show that both the curve and its AUC vary across held-out folds.](assets/roc-cross-validation.png){fig-align="center" width="72%" fig-alt="ROC curves across cross-validation folds with mean and variation band"}

*Image: [scikit-learn, ROC with cross validation](https://scikit-learn.org/stable/auto_examples/model_selection/plot_roc_crossval.html). Fold variation is descriptive; correlated folds require care in formal inference.*

#### **Bootstrap Confidence Intervals**

The nonparametric bootstrap approximates repeated sampling from an unknown population by resampling observed units with replacement. For a statistic $T(D)$, generate bootstrap datasets $D^{*(1)},\ldots,D^{*(B)}$ and compute

$$
T^{*(b)}=T(D^{*(b)}).
$$

A simple $(1-\alpha)$ percentile interval uses the empirical $\alpha/2$ and $1-\alpha/2$ quantiles of $\{T^{*(b)}\}$. Bias-corrected and accelerated intervals can improve coverage in some settings, but no interval repairs a nonrepresentative test sample.

The resampling unit is crucial:

- independent rows permit row bootstrap;
- repeated observations require resampling whole subjects or groups;
- temporal dependence calls for a block bootstrap or a time-series model;
- spatial clusters should remain intact.

If model predictions on a locked test set are resampled without retraining, the interval describes **test-sample uncertainty conditional on the fitted models**. To include training and selection variability, the full training and selection procedure must be repeated inside each bootstrap or outer resampling loop, which is much more expensive.

![Paired bootstrap resamples the same test units for both models and forms a distribution of metric differences.](assets/bootstrap-paired-comparison.svg){fig-align="center" width="100%" fig-alt="Paired bootstrap workflow from locked predictions through joint resampling to a distribution of metric differences"}

<details>
<summary><strong>Python example: a paired bootstrap interval for the difference in ROC AUC</strong></summary>

```python
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=73
)

model_a = make_pipeline(
    StandardScaler(), LogisticRegression(max_iter=2000, random_state=73)
).fit(X_train, y_train)
model_b = RandomForestClassifier(
    n_estimators=250, min_samples_leaf=3, random_state=73, n_jobs=1
).fit(X_train, y_train)

probability_a = model_a.predict_proba(X_test)[:, 1]
probability_b = model_b.predict_proba(X_test)[:, 1]
observed_difference = roc_auc_score(y_test, probability_b) - roc_auc_score(y_test, probability_a)

rng = np.random.default_rng(73)
differences = []
for _ in range(2000):
    index = rng.integers(0, len(y_test), size=len(y_test))
    if np.unique(y_test[index]).size < 2:
        continue
    differences.append(
        roc_auc_score(y_test[index], probability_b[index])
        - roc_auc_score(y_test[index], probability_a[index])
    )

lower, upper = np.quantile(differences, [0.025, 0.975])
print("observed AUC difference (B - A):", round(observed_difference, 4))
print("paired bootstrap 95% interval:  ", tuple(np.round([lower, upper], 4)))
print("valid bootstrap resamples:      ", len(differences))
```

</details>

#### **Paired Tests and Multiple Comparisons**

Two models evaluated on the same cases produce paired outcomes. The pairing removes variation due to one test sample being easier than another and should be preserved in tests and resampling.

For two hard classifiers, **McNemar's test** ignores cases both models handle identically and uses only discordant counts:

| | Model B correct | Model B wrong |
|---|---:|---:|
| Model A correct | both correct | $b$ |
| Model A wrong | $c$ | both wrong |

Under the null that neither model is more likely to be correct, $b$ conditional on $b+c$ follows $\operatorname{Binomial}(b+c,0.5)$. The exact binomial test is preferable when discordant counts are small. For continuous per-case losses, a paired permutation or sign-flip test can assess whether an observed aggregate difference is unusual under exchangeability.

<details>
<summary><strong>Python example: exact McNemar comparison from paired predictions</strong></summary>

```python
import numpy as np
from scipy.stats import binomtest

y_true = np.array([1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0])
prediction_a = np.array([1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0])
prediction_b = np.array([1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0])

correct_a = prediction_a == y_true
correct_b = prediction_b == y_true
b = int(np.sum(correct_a & ~correct_b))  # A alone is correct
c = int(np.sum(~correct_a & correct_b))  # B alone is correct

test = binomtest(k=min(b, c), n=b + c, p=0.5, alternative="two-sided")
print("A correct / B wrong:", b)
print("A wrong / B correct:", c)
print("exact two-sided p-value:", round(test.pvalue, 4))
```

</details>

Cross-validation adds dependence because training sets overlap. Treating fold scores as independent in an ordinary paired $t$-test underestimates variance. Corrected resampled tests inflate the variance using the test-to-train ratio; repeated nested CV or a carefully designed permutation test may be more appropriate. The experimental unit remains the independently sampled dataset or group, not every prediction emitted from overlapping fitted models.

When $M$ hypotheses are tested, the probability of at least one false discovery increases. Bonferroni compares each $p$-value with $\alpha/M$ and controls family-wise error conservatively. **Holm's procedure** sorts $p_{(1)}\leq\cdots\leq p_{(M)}$ and sequentially compares $p_{(j)}$ with $\alpha/(M-j+1)$; it controls family-wise error while usually being less conservative. False-discovery-rate procedures answer a different question and may be suitable for exploratory screening.

<details>
<summary><strong>Python example: apply Holm correction to several planned comparisons</strong></summary>

```python
import numpy as np

names = np.array(["B vs A", "C vs A", "D vs A", "E vs A", "F vs A"])
p_values = np.array([0.004, 0.018, 0.031, 0.070, 0.220])
alpha = 0.05

order = np.argsort(p_values)
adjusted_sorted = np.maximum.accumulate(
    (len(p_values) - np.arange(len(p_values))) * p_values[order]
)
adjusted_sorted = np.minimum(adjusted_sorted, 1.0)
adjusted = np.empty_like(adjusted_sorted)
adjusted[order] = adjusted_sorted

# Holm rejects sequentially until the first failed threshold.
still_rejecting = True
rejected = np.zeros(len(p_values), dtype=bool)
for rank, index in enumerate(order):
    threshold = alpha / (len(p_values) - rank)
    still_rejecting = still_rejecting and (p_values[index] <= threshold)
    rejected[index] = still_rejecting

for name, raw, corrected, reject in zip(names, p_values, adjusted, rejected):
    print(f"{name:6s}: raw={raw:.3f}, Holm-adjusted={corrected:.3f}, reject={reject}")
```

</details>

Tests planned before observing results have a clearer interpretation than tests chosen after searching many models, seeds, subsets, and metrics. Selective reporting is itself a multiple-comparison process, even when only one final $p$-value appears in the article.

#### **Practical versus Statistical Significance**

Statistical significance concerns compatibility with a null model under assumptions. Practical significance concerns whether an effect is large enough to matter. With enough data, a negligible improvement can have a small $p$-value; with little data, a valuable improvement can remain uncertain.

Define a minimum practically important difference $\delta>0$ in advance. For utility difference $\Delta=U_B-U_A$:

- $\Delta>\delta$ is meaningfully better;
- $|\Delta|\leq\delta$ is practically equivalent or inconclusive within a **region of practical equivalence** (ROPE);
- $\Delta<-\delta$ is meaningfully worse.

Confidence intervals quantify a range of compatible effect sizes. An interval crossing zero does not prove equivalence; equivalence requires sufficiently precise evidence inside a prespecified margin. The decision should also include training cost, inference latency, memory, maintenance burden, subgroup effects, and failure severity.

![A comparison distribution can be interpreted relative to a region of practical equivalence rather than only a zero-difference line.](assets/model-comparison-rope.png){fig-align="center" width="76%" fig-alt="Distribution of model performance differences divided into worse, equivalent, and better regions"}

*Image: [scikit-learn, Statistical comparison of models using grid search](https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_stats.html). The Bayesian ROPE display illustrates probability mass assigned to practically worse, equivalent, and better regions.*

<details>
<summary><strong>Python example: summarize paired uncertainty using a practical equivalence margin</strong></summary>

```python
import numpy as np

rng = np.random.default_rng(79)
n_cases = 500

# Per-case utility is paired because both systems face the same cases.
utility_a = rng.normal(loc=0.72, scale=0.18, size=n_cases)
utility_b = utility_a + rng.normal(loc=0.012, scale=0.06, size=n_cases)
margin = 0.01

bootstrap_difference = []
for _ in range(5000):
    index = rng.integers(0, n_cases, size=n_cases)
    bootstrap_difference.append(np.mean(utility_b[index] - utility_a[index]))
bootstrap_difference = np.asarray(bootstrap_difference)

print("estimated mean difference:", round(np.mean(utility_b - utility_a), 4))
print("95% interval:", tuple(np.round(np.quantile(bootstrap_difference, [0.025, 0.975]), 4)))
print("probability meaningfully worse:", round(np.mean(bootstrap_difference < -margin), 3))
print("probability practically equivalent:", round(np.mean(np.abs(bootstrap_difference) <= margin), 3))
print("probability meaningfully better:", round(np.mean(bootstrap_difference > margin), 3))
```

</details>

**Uncertainty comparison.** Bootstrap intervals are flexible when the sampling unit is respected. Paired tests exploit shared cases; cross-validation comparisons require corrections for dependence. Multiple-comparison procedures address a family of planned claims, not undisclosed experimentation. Effect sizes, intervals, and practical margins should lead interpretation; a binary significance label should not.


### **Error Analysis and Data Slicing**

An aggregate metric determines whether a system is acceptable on average; **error analysis** investigates where and why it fails. The goal is not to collect memorable mistakes but to discover reproducible patterns that lead to a testable hypothesis about data, labels, representation, model, threshold, or distribution shift.

#### **Confusion Patterns and Residual Analysis**

For multiclass classification, a confusion matrix $C$ has entries

$$
C_{ij}=\#\{n:y_n=i,\widehat y_n=j\}.
$$

Raw counts reveal operational volume. Row normalization estimates $\Pr(\widehat Y=j\mid Y=i)$ and exposes where each true class goes; column normalization estimates $\Pr(Y=i\mid\widehat Y=j)$ and exposes the composition of each predicted class. Both should include supports because a dramatic percentage based on three cases is weak evidence.

Useful inspection categories include:

- **data problems:** duplicates, missing context, corrupted measurements, preprocessing mismatch;
- **label problems:** ambiguity, inconsistent annotation, stale labels, ontology mismatch;
- **model problems:** underfitting, spurious features, poor calibration, insufficient context;
- **shift problems:** new subpopulations, temporal drift, changed collection process;
- **decision problems:** threshold or cost assumptions that do not match operations.

Cases should be sampled systematically from high-volume confusion cells, high-confidence errors, and low-confidence correct predictions. Reading only the most surprising failures exaggerates anecdotes and cannot estimate prevalence.

<details>
<summary><strong>Python example: rank multiclass confusion patterns by count and conditional rate</strong></summary>

```python
import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=83
)
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2500, random_state=83),
).fit(X_train, y_train)
prediction = model.predict(X_test)

counts = confusion_matrix(y_test, prediction)
rates = counts / counts.sum(axis=1, keepdims=True)

patterns = []
for true_class in range(counts.shape[0]):
    for predicted_class in range(counts.shape[1]):
        if true_class != predicted_class and counts[true_class, predicted_class] > 0:
            patterns.append(
                (counts[true_class, predicted_class], rates[true_class, predicted_class],
                 true_class, predicted_class)
            )

print("test accuracy:", round(np.mean(prediction == y_test), 3))
print("largest off-diagonal confusion cells:")
for count, rate, true_class, predicted_class in sorted(patterns, reverse=True)[:6]:
    print(
        f"  true {true_class} -> predicted {predicted_class}: "
        f"count={count}, within-true-class rate={rate:.3f}"
    )
```

</details>

For regression, residuals $e_i=y_i-\widehat y_i$ preserve direction. Analyze their center, spread, tails, and relation to predictions, features, time, and groups. A positive residual means underprediction under this convention. Useful checks include residual-versus-prediction plots, binned residual summaries, quantile coverage, autocorrelation, and errors at operationally important target ranges.

<details>
<summary><strong>Python example: binned residuals expose a missing nonlinear relationship</strong></summary>

```python
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(89)
X = rng.uniform(-3, 3, size=(1000, 1))
y = 1.0 + 0.7 * X[:, 0] + 0.9 * X[:, 0] ** 2 + rng.normal(0, 0.7, size=1000)

# A straight line cannot represent the quadratic component.
model = LinearRegression().fit(X, y)
prediction = model.predict(X)
residual = y - prediction

diagnostic = pd.DataFrame({"x": X[:, 0], "residual": residual})
diagnostic["x_bin"] = pd.cut(diagnostic["x"], bins=np.linspace(-3, 3, 7), include_lowest=True)
summary = diagnostic.groupby("x_bin", observed=True)["residual"].agg(["count", "mean", "std"])

print("overall mean residual:", round(residual.mean(), 4))
print(summary.round(3))
```

</details>

The overall residual mean is near zero because an intercept absorbs average bias, yet the binned means reveal a U-shaped misspecification. This illustrates why one scalar cannot certify model adequacy.

#### **Subgroup, Temporal, and Stress-Test Slices**

A **slice** is a subset defined by a deployment-relevant condition. High-stakes slices should be specified before final evaluation; exploratory slices can generate hypotheses but require fresh confirmation. Important families include:

- subgroup slices by user, device, region, language, or acquisition channel;
- intersectional slices combining attributes when operationally justified;
- temporal cohorts that reveal drift, seasonality, or launch effects;
- target-range and confidence slices;
- stress tests with missing features, measurement noise, compression, latency constraints, or plausible perturbations.

![Aggregate performance can conceal subgroup, temporal, and stress-test failures; analysis should move from detection to inspection, intervention, and retesting.](assets/error-slicing.svg){fig-align="center" width="100%" fig-alt="Overall score leading to a slice table and a structured error analysis loop"}

Every slice needs its sample size, metric definition, and uncertainty. Tiny slices produce unstable rankings and invite false discoveries. Hierarchical models or partial pooling may be preferable to dozens of independent estimates. A slice must also correspond to a legitimate use case; searching arbitrary partitions until one looks bad is not robust evaluation.

<details>
<summary><strong>Python example: an aggregate score hides a low-performing subgroup</strong></summary>

```python
import numpy as np
import pandas as pd

def wilson_interval(successes, n, confidence_z=1.96):
    proportion = successes / n
    denominator = 1 + confidence_z**2 / n
    centre = (proportion + confidence_z**2 / (2 * n)) / denominator
    radius = confidence_z * np.sqrt(
        proportion * (1 - proportion) / n + confidence_z**2 / (4 * n**2)
    ) / denominator
    return centre - radius, centre + radius

rng = np.random.default_rng(97)
groups = np.array(["majority"] * 900 + ["minority"] * 100)
correct = np.concatenate([
    rng.binomial(1, 0.94, size=900),
    rng.binomial(1, 0.68, size=100),
])
frame = pd.DataFrame({"group": groups, "correct": correct})

print("overall accuracy:", round(frame["correct"].mean(), 3))
for group, subset in frame.groupby("group"):
    successes, n = int(subset["correct"].sum()), len(subset)
    lower, upper = wilson_interval(successes, n)
    print(
        f"{group:8s}: n={n}, accuracy={successes / n:.3f}, "
        f"Wilson 95% interval=[{lower:.3f}, {upper:.3f}]"
    )
```

</details>

Stress tests should perturb inputs in a plausible and reproducible way. Their result is a performance curve over severity, not proof of robustness to every possible corruption. Perturbations must respect semantics; random pixel noise, for example, may not represent blur, lighting changes, or a different camera.

<details>
<summary><strong>Python example: measure degradation under increasing feature noise</strong></summary>

```python
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, stratify=y, random_state=101
)
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=101),
).fit(X_train, y_train)

rng = np.random.default_rng(101)
feature_scale = X_train.std(axis=0, ddof=1)

for severity in [0.0, 0.1, 0.25, 0.5, 1.0]:
    corrupted = X_test + rng.normal(size=X_test.shape) * feature_scale * severity
    probability = model.predict_proba(corrupted)[:, 1]
    prediction = probability >= 0.5
    print(
        f"noise severity={severity:>4.2f}",
        "accuracy =", round(accuracy_score(y_test, prediction), 3),
        "log loss =", round(log_loss(y_test, probability), 3),
    )
```

</details>

Error analysis must preserve test discipline. Once a test-set pattern motivates a model change, that historical test set has influenced development. The change should be validated on development data and confirmed on new untouched data, a prospectively collected cohort, or a new temporal window.


### **Reproducibility and Result Reporting**

Reproducibility is the ability to reconstruct the data, procedure, and numerical result closely enough to audit a claim. It is not achieved by writing `random_state=42` in one estimator. Randomness can enter splitting, sampling, initialization, parallel reductions, data loading, augmentation, hardware kernels, and external services.

#### **Seeds, Versions, and Experimental Configuration**

A reproducible experiment records at least:

- immutable identifiers or checksums for raw data, labels, and split assignments;
- source-control commit and a description of uncommitted changes;
- environment lockfile, library versions, runtime, operating system, and relevant hardware;
- preprocessing, feature schema, estimator, hyperparameters, threshold, metric, and averaging rules;
- every random seed and the policy used to derive child seeds;
- search space, budget, early-stopping rule, failed trials, and selection criterion;
- output artifacts such as predictions, logs, timing, and model files.

One master seed can deterministically create independent child streams through `numpy.random.SeedSequence`. Reusing one pseudo-random sequence everywhere can accidentally couple augmentation, splitting, and initialization. A serialized configuration should be canonicalized before hashing so that key order does not change its identifier.

<details>
<summary><strong>Python example: create an auditable configuration fingerprint and child seeds</strong></summary>

```python
import hashlib
import json
import platform
from importlib.metadata import version
import numpy as np

configuration = {
    "data_version": "breast-cancer-v1",
    "split": {"method": "stratified_holdout", "test_size": 0.20},
    "pipeline": {
        "scaler": "StandardScaler",
        "model": "LogisticRegression",
        "C": 1.0,
    },
    "selection_metric": "negative_log_loss",
    "decision_threshold": 0.50,
    "master_seed": 107,
}

canonical = json.dumps(configuration, sort_keys=True, separators=(",", ":"))
fingerprint = hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:16]

seed_sequence = np.random.SeedSequence(configuration["master_seed"])
split_seed, model_seed, bootstrap_seed = [
    int(child.generate_state(1, dtype=np.uint32)[0])
    for child in seed_sequence.spawn(3)
]

print("experiment fingerprint:", fingerprint)
print("child seeds:", {"split": split_seed, "model": model_seed, "bootstrap": bootstrap_seed})
print("environment:", {
    "python": platform.python_version(),
    "numpy": version("numpy"),
    "scikit-learn": version("scikit-learn"),
})
```

</details>

Determinism and reproducibility are related but distinct. Bitwise determinism may require slower kernels and fixed thread scheduling, while scientific reproducibility asks whether the conclusion survives reasonable reruns and implementation details. If an algorithm is intentionally stochastic, report its distribution across prespecified seeds rather than hiding it behind one favorable run.

#### **Reporting Mean, Variation, Cost, and Limitations**

A result table should report enough context to interpret each number. At minimum include the evaluation population and period, split unit and protocol, metric definition, point estimate, uncertainty or variation, number of evaluated units, threshold, selection budget, and computational cost. Distinguish:

- **standard deviation across runs**, which describes run-to-run dispersion;
- **standard error of a mean**, which describes uncertainty in an estimated mean under a sampling model;
- **confidence interval**, whose coverage depends on the stated resampling assumptions;
- **cross-validation fold range**, which is descriptive because folds are correlated.

Compute is part of the comparison. A 0.2 percentage-point gain may be unattractive if training cost rises by 100 times or latency violates a service objective. Report wall-clock time with hardware and concurrency, peak memory when relevant, model size, and inference throughput or latency distribution.

<details>
<summary><strong>Python example: summarize repeated evaluation with score and fit-time variation</strong></summary>

```python
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_breast_cancer(return_X_y=True)
procedure = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2000, random_state=109),
)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=109)
result = cross_validate(
    procedure,
    X,
    y,
    cv=cv,
    scoring={"roc_auc": "roc_auc", "log_loss": "neg_log_loss"},
    n_jobs=1,
)

auc = result["test_roc_auc"]
log_loss = -result["test_log_loss"]
print("evaluations:", len(auc))
print("ROC AUC mean and descriptive SD:", round(auc.mean(), 4), round(auc.std(ddof=1), 4))
print("log loss mean and descriptive SD:", round(log_loss.mean(), 4), round(log_loss.std(ddof=1), 4))
print("fit time mean and maximum (s):", round(result["fit_time"].mean(), 4),
      round(result["fit_time"].max(), 4))
print("Note: fold scores overlap; this SD is not an independent-sample confidence interval.")
```

</details>

A concise model-selection report can follow this workflow:

1. State the deployment unit, population, outcome, horizon, and action.
2. Freeze inclusion criteria, split structure, metrics, and practical effect threshold.
3. Isolate test units before exploratory modeling.
4. Put all learned preprocessing inside the training pipeline.
5. Compare a simple baseline and candidate procedures using validation or inner CV.
6. Tune probability calibration and decision thresholds only on development data.
7. Freeze code, configuration, environment, and decision rule.
8. Evaluate once on the locked test set with uncertainty and prespecified slices.
9. Report effect sizes, variation, resource cost, failures, and limitations.
10. Monitor the deployed system and begin a new evaluation cycle when the target process changes.

The final limitations section should name unsupported populations, unresolved confounding in data collection, uncertain labels, weak slices, possible leakage channels, sensitivity to seeds or hyperparameters, and the difference between offline metrics and deployment outcomes. Clear limitations narrow a claim to what the experiment actually supports; they do not weaken a well-designed study.

**Chapter summary.** Reliable evaluation is an experimental design problem. Structure-aware splits define what “unseen” means; task-aligned metrics define success; calibration and thresholds connect scores to decisions; search procedures consume validation evidence; uncertainty and paired comparisons qualify apparent improvements; error slices expose structured failures; and reproducible reporting makes the entire claim auditable.
